# poblar — ejecución en Colab

Este cuaderno instala la librería y la invoca. Los criterios están en el archivo de configuración; los productos, los controles y el manifiesto quedan en la carpeta de salida.

1. Completar los parámetros de la celda 3.
2. Ejecutar las celdas en orden.
3. Revisar `controles.csv` y `balance.csv` en la carpeta de la ejecución.

Entorno: Colab CPU. Las etapas 2 y 3 necesitan las herramientas de GDAL (la celda 1 las instala si faltan) y espacio en disco local: los rasters se procesan en `/content/trabajo` y al final se copian a Drive.

In [ ]:
# 1. Instalación
# Se clona el repositorio y se instala desde el clon: el código y las configuraciones
# son de la misma versión, y la ejecución registra el commit en entorno.txt.
# Para reproducir una versión exacta, indicar una etiqueta o un commit en VERSION.
VERSION = 'main'
!rm -rf /content/poblar_repo
!git clone -q https://github.com/AugustoBellanich/poblar /content/poblar_repo
!git -C /content/poblar_repo checkout -q {VERSION}
!pip install -q -e /content/poblar_repo
# La instalación editable se activa recién al reiniciar el intérprete; con esto
# la librería se puede importar en la misma sesión, sin reiniciar.
import sys, importlib
if '/content/poblar_repo' not in sys.path:
    sys.path.insert(0, '/content/poblar_repo')
importlib.invalidate_caches()
# Si en esta sesión ya se había importado una versión anterior, se descarta para usar la nueva.
for _m in [m for m in sys.modules if m == 'poblar' or m.startswith('poblar.')]:
    del sys.modules[_m]
!git -C /content/poblar_repo log -1 --format='Versión: %h — %s (%cs)'

# Herramientas de GDAL (gdalwarp, gdal_rasterize) para las etapas 2 y 3
!which gdalwarp > /dev/null || (apt-get -qq update && apt-get -qq install -y gdal-bin > /dev/null)
!gdalinfo --version


In [ ]:
# 2. Montaje de Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Parámetros de la ejecución
from pathlib import Path

ANIO    = 2010                                                   # 2010 o 2022
CONFIG  = Path(f'/content/poblar_repo/aplicaciones/prais2026/configs/{ANIO}.yaml')
INSUMOS = Path('/content/drive/MyDrive/PRAIS/poblar/insumos')   # tabla_censoAAAA.xlsx, radios_AAAA.zip, regiones_PRAIS.gpkg, MB_prais_2015_100m.tif (2010) o MB_prais_2023_100m.tif (2022),
                                                                 # degradacion_referencia_2010.tif o degradacion_reporte_2022.tif
SALIDA  = Path('/content/drive/MyDrive/PRAIS/poblar/ejecuciones')
ETAPAS  = ['insumos', 'radios', 'grilla', 'dasimetria', 'cruce', 'publicacion', 'tablas']
# Para retomar una ejecución existente, indicar solo las etapas que faltan (por ejemplo ['tablas']).
TRABAJO = Path('/content/trabajo')                              # disco local de Colab; al terminar se copia a SALIDA

# Verificación previa: que estén los insumos que pide la configuración
import yaml
cfg = yaml.safe_load(CONFIG.read_text(encoding='utf-8'))
faltan = [i['ruta'] for i in cfg['insumos'].values() if not (INSUMOS / i['ruta']).exists()]
if faltan:
    raise FileNotFoundError(f'Faltan insumos en {INSUMOS}: {faltan}')
print('Insumos completos:', [i['ruta'] for i in cfg['insumos'].values()])

In [ ]:
# 4. Ejecución
# Si un control bloqueante falla, la ejecución se detiene y se informa el motivo;
# los controles y el balance quedan igual en la carpeta para revisarlos.
from poblar import ejecutar
from poblar.trazabilidad import ControlFallido

try:
    ejecucion = ejecutar(CONFIG, insumos=INSUMOS, salida=SALIDA, etapas=ETAPAS,
                         trabajo=TRABAJO)
    CARPETA = ejecucion.carpeta
    print('\nEjecución completa:', CARPETA)
except ControlFallido as e:
    CARPETA = e.carpeta
    print('\nLa ejecución se detuvo:', e)
    print('Carpeta:', CARPETA)

In [ ]:
# 5. Controles y balance
import pandas as pd
pd.set_option('display.max_colwidth', 120)
controles = pd.read_csv(CARPETA / 'controles.csv')
print(f"Controles: {len(controles)} | sin cumplir: {(~controles['ok']).sum()}")
display(controles[['etapa', 'nombre', 'ok', 'detalle', 'bloqueante']])
display(pd.read_csv(CARPETA / 'balance.csv'))

## Comparación con la ejecución de referencia (opcional)

Compara `radios.parquet` con la caché del C2 original (`02_radios_curados_<año>.parquet`, en `PRAIS/Cuadernos/C2/<año>/cache`). La caché del 25/10/2026 se generó con la unión `left`, por lo que tiene un radio más (sin dato censal, población 0).

In [ ]:
# 6. Comparación radio por radio
from poblar.verificar import comparar_radios

REF = Path(f'/content/drive/MyDrive/PRAIS/Cuadernos/C2/{ANIO}/cache/02_radios_curados_{ANIO}.parquet')
NUEVO = CARPETA / '1_radios' / 'radios.parquet'
if not NUEVO.exists():
    print('La ejecución no generó radios.parquet: revisar los controles de la celda 5.')
elif not REF.exists():
    print('No se encontró la referencia:', REF)
else:
    display(comparar_radios(NUEVO, REF))

## Comparación de los rasters (opcional)

Compara píxel por píxel cada raster de `3_dasimetria` con el del C2 original (`PRAIS/Cuadernos/C2/<año>/procesado`). Si la grilla, la máscara y los valores coinciden, la etapa 3 reproduce el método original.


In [ ]:
# 7. Comparación píxel por píxel con el C2 original
from poblar.verificar import comparar_raster

REF_DIR = Path(f'/content/drive/MyDrive/PRAIS/Cuadernos/C2/{ANIO}/procesado')
NUEVO_DIR = CARPETA / '3_dasimetria'
nuevos = sorted(NUEVO_DIR.glob(f'*_{ANIO}.tif')) if NUEVO_DIR.exists() else []
if not nuevos:
    print('La ejecución no generó rasters de la etapa 3: revisar los controles de la celda 5.')
resumen = []
for r in nuevos:
    ref = REF_DIR / r.name
    if not ref.exists():
        resumen.append({'raster': r.name, 'resultado': 'sin referencia'})
        continue
    t = comparar_raster(r, ref)
    resumen.append({'raster': r.name, 'resultado': 'igual' if t['igual'].all() else 'DISTINTO',
                    'suma_nueva': t.loc[t.comparacion == 'suma', 'nuevo'].item(),
                    'suma_referencia': t.loc[t.comparacion == 'suma', 'referencia'].item(),
                    'dif_max_pixel': t.iloc[-1]['nuevo']})
    if not t['igual'].all():
        print(r.name); display(t)
display(pd.DataFrame(resumen))


## Comparación del cruce con el C3 (opcional)

Compara `4_cruce/cruce_largo.csv` con la tabla larga del C3 (`exposicion_degradacion_<año>_largo.csv`), fila por fila (región × ámbito × clase × variable). Por defecto usa la reproducción del 26/10 (`PRAIS/Cuadernos/C3/<año>/procesado`), que se hizo sobre el C2 con la unión `left`: puede haber diferencias mínimas en una zona.


In [ ]:
# 8. Comparación del cruce con el C3
from poblar.verificar import comparar_cruce

REF_C3 = Path(f'/content/drive/MyDrive/PRAIS/Cuadernos/C3/{ANIO}/procesado/exposicion_degradacion_{ANIO}_largo.csv')
NUEVO_C3 = CARPETA / '4_cruce' / 'cruce_largo.csv'
if not NUEVO_C3.exists():
    print('La ejecución no generó el cruce: revisar los controles de la celda 5.')
elif not REF_C3.exists():
    print('No se encontró la referencia:', REF_C3)
else:
    display(comparar_cruce(NUEVO_C3, REF_C3))
    r = pd.read_csv(CARPETA / '4_cruce' / 'resumen_expuesta.csv')
    display(r[r.region == 'NACIONAL'])


## Comparación de la publicación con el C5 (opcional)

Compara cada raster de `5_publicacion` con el equivalente de la reproducción del C5 (`PRAIS/Cuadernos/C5/<año>`, nombres `<producto>_dasimetrico_<año>_WGS84.tif`).


In [ ]:
# 9. Comparación de la publicación con el C5
from poblar.verificar import comparar_raster

REF_C5 = Path(f'/content/drive/MyDrive/PRAIS/Cuadernos/C5/{ANIO}')
PUB = CARPETA / '5_publicacion'
if not (PUB / 'publicacion.csv').exists():
    print('La ejecución no generó la publicación: revisar los controles de la celda 5.')
else:
    pub = pd.read_csv(PUB / 'publicacion.csv')
    display(pub)
    resumen = []
    for _, f in pub.iterrows():
        nombre_c5 = (f'poblacion_dasimetrica_{ANIO}_WGS84.tif' if f.producto == 'poblacion_dasimetrica'
                     else f'{f.producto}_dasimetrico_{ANIO}_WGS84.tif')
        ref = REF_C5 / nombre_c5
        if not ref.exists():
            resumen.append({'archivo': f.archivo, 'resultado': 'sin referencia'}); continue
        t = comparar_raster(PUB / f.archivo, ref)
        resumen.append({'archivo': f.archivo, 'resultado': 'igual' if t['igual'].all() else 'DISTINTO',
                        'suma_nueva': t.loc[t.comparacion == 'suma', 'nuevo'].item(),
                        'suma_c5': t.loc[t.comparacion == 'suma', 'referencia'].item(),
                        'pixeles_mascara': t.loc[t.comparacion.str.startswith('píxeles con distinta'), 'nuevo'].item(),
                        'dif_max_pixel': t.iloc[-1]['nuevo']})
    display(pd.DataFrame(resumen))


## Tablas de los dos períodos y comparación con el informe

Correr después de ejecutar la etapa `tablas` en los dos años. Une las tablas de 2010 y 2022 (columnas por período) y compara cada cifra de las Tablas 6 a 11 del informe con las de poblar, indicando qué fórmula reproduce cada porcentaje publicado.


In [ ]:
# 10. Tablas 2010–2022 y comparación con el informe
from poblar.tablas import comparar_informe, unir_periodos

EJEC = {2010: SALIDA / 'prais2026_2010', 2022: SALIDA / 'prais2026_2022'}
DEST = SALIDA / 'prais2026_tablas'
unidas = unir_periodos(EJEC, DEST)
for nombre, t in unidas.items():
    print(nombre); display(t)

REF = Path('/content/poblar_repo/aplicaciones/prais2026/referencias/informe_tablas_6_11.csv')
comp = comparar_informe(EJEC, REF)
comp.to_csv(DEST / 'comparacion_informe.csv', index=False)
print('Cifras con |diferencia| > 1 persona:', int((comp.dif_n.abs() > 1).sum()), 'de', len(comp))
display(comp.groupby(['tabla', 'anio', 'ambito'])['formula_que_coincide'].value_counts().rename('celdas'))
display(comp)
